In [2]:
# =============================================================================
# SECTION 0 - IMPORT CONFIG + SETUP
# =============================================================================

In [1]:
import sys
from pathlib import Path

# Get the current working directory (location of this notebook)
# The notebook is located in: helper_functions_normalization/
current_dir = Path.cwd()

# The project root is one level above the notebook directory
project_root = current_dir.parent

# Add the project root to the Python module search path
# This allows imports like: from config.config import POSITION_KPIS
sys.path.insert(0, str(project_root))

In [2]:
from __future__ import annotations

from dataclasses import dataclass
from pathlib import Path
from typing import Any

import pandas as pd
import numpy as np


# --- Import all normalization settings from config ---
from config.config import (
    POSITION_KPIS,
    POSITION_MAPPING,
    AGE_GROUPS,
    STAGE_BY_AGEGROUP,
    STAGE_MINUTES,
    MINUTES_PERCENTILE_Q,
    MINUTES_COLUMN,
    PEER_GROUP_COLS
)

In [3]:
# =============================================================================
# SECTION 1 — AGE GROUPING HELPERS
# =============================================================================

In [4]:
def parse_age_group_label(label: str) -> tuple[int | None, int | None]:
    """
    Convert an AGE_GROUPS label into a (min_age, max_age) tuple.

    Supported formats:
    - "<21"   -> (None, 20)   meaning: age <= 20
    - "21-23" -> (21, 23)     meaning: 21 <= age <= 23
    - "27+"   -> (27, None)   meaning: age >= 27

    Examples
    --------
    parse_age_group_label("<21")   -> (None, 20)
    parse_age_group_label("21-23") -> (21, 23)
    parse_age_group_label("27+")   -> (27, None)
    """
    label = label.strip()

    # Case 1: "<N" (strictly less than N, so max_age = N-1)
    if label.startswith("<"):
        n = int(label[1:])
        return (None, n - 1)

    # Case 2: "N+"
    if label.endswith("+"):
        n = int(label[:-1])
        return (n, None)

    # Case 3: "A-B"
    if "-" in label:
        a, b = label.split("-")
        return (int(a), int(b))

    raise ValueError(f"Unsupported AGE_GROUPS label format: {label}")


def make_age_group(age: float, age_groups: list[str]) -> str:
    """
    Assign an age group label based on a list of AGE_GROUPS labels.
    The first matching rule is used.

    Notes
    -----
    - Returns NaN if age is missing or doesn't match any configured bucket.
    - This is intentional to avoid silently assigning the wrong group.

    Example
    -------
    AGE_GROUPS = ["<21", "21-23", "24-26", "27+"]

    make_age_group(20, AGE_GROUPS) -> "<21"
    make_age_group(22, AGE_GROUPS) -> "21-23"
    make_age_group(30, AGE_GROUPS) -> "27+"
    """
    if pd.isna(age):
        return np.nan

    for label in age_groups:
        min_age, max_age = parse_age_group_label(label)

        min_ok = True if min_age is None else age >= min_age
        max_ok = True if max_age is None else age <= max_age

        if min_ok and max_ok:
            return label

    return np.nan


def make_player_stage(age_group: str, stage_by_agegroup: dict) -> str:
    """
    Map an age_group label to a player_stage using a config mapping.

    Example
    -------
    STAGE_BY_AGEGROUP = {
        "<21": "Early Talent",
        "21-23": "Developing Talent",
        "24-26": "Prime Transition",
        "27+": "Established Player"
    }

    make_player_stage("21-23", STAGE_BY_AGEGROUP) -> "Developing Talent"
    """
    if pd.isna(age_group):
        return np.nan
    return stage_by_agegroup.get(age_group, np.nan)


def add_age_group_and_stage(
    df: pd.DataFrame,
    age_col: str,
    age_groups: list[str],
    stage_by_agegroup: dict,
    out_age_group_col: str = "age_group",
    out_stage_col: str = "player_stage",
) -> pd.DataFrame:
    """
    Add age_group and player_stage columns.

    Parameters
    ----------
    age_col:
        Column in df that holds the player's age (e.g. "Age")
    out_age_group_col / out_stage_col:
        Output column names (defaults match your config usage).
    """
    df = df.copy()
    df[out_age_group_col] = df[age_col].apply(lambda x: make_age_group(x, age_groups))
    df[out_stage_col] = df[out_age_group_col].apply(lambda ag: make_player_stage(ag, stage_by_agegroup))
    return df

In [5]:
# =============================================================================
# SECTION 2 — NUMERIC + IMPUTATION + Z-SCORES
# =============================================================================

In [6]:
def force_numeric(df: pd.DataFrame, cols: list[str]) -> pd.DataFrame:
    """
    Convert selected columns to numeric. Non-parsable values become NaN.

    Why this matters
    ----------------
    CSVs often contain numeric values stored as strings or with invalid tokens.
    For normalization, you need stable numeric dtype.

    Example
    -------
    df["x"] = ["1", "2", "oops"]
    force_numeric(df, ["x"]) -> df["x"] becomes [1.0, 2.0, NaN]
    """
    df = df.copy()
    for c in cols:
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors="coerce")
    return df


def impute_peer_group_median(
    df: pd.DataFrame,
    feature_cols: list[str],
    group_cols: list[str]
) -> pd.DataFrame:
    """
    Two-step imputation strategy:
    1) Fill missing values using peer-group medians (defined by group_cols)
    2) If still missing (e.g., all NaN within a peer group), fallback to overall median

    Example
    -------
    group_cols = ["League_Name", "age_group"]
    For each KPI column, missing values are filled per (League_Name, age_group).

    Notes
    -----
    - This keeps the imputation consistent with your normalization grouping.
    - Fallback prevents "stuck NaNs" when an entire peer group has no data for a KPI.
    """
    df = df.copy()

    # First pass: median within peer group
    for col in feature_cols:
        peer_median = df.groupby(group_cols)[col].transform("median")
        df[col] = df[col].fillna(peer_median)

    # Second pass: overall median for any remaining missing values
    missing_after = df[feature_cols].isna().sum()
    cols_still_missing = missing_after[missing_after > 0].index.tolist()

    for col in cols_still_missing:
        overall_median = df[col].median(skipna=True)
        df[col] = df[col].fillna(overall_median)

    return df


def z_score(series: pd.Series) -> pd.Series:
    """
    Compute z-scores safely.

    Returns
    -------
    - (x - mean) / std
    - If std is 0 or missing -> returns NaN (to avoid division by zero)

    Example
    -------
    series = [10, 10, 10]
    std = 0 -> all z-scores become NaN
    """
    mean = series.mean(skipna=True)
    std = series.std(skipna=True, ddof=0)

    if std == 0 or pd.isna(std):
        return np.nan

    return (series - mean) / std


def add_peer_group_zscores(
    df: pd.DataFrame,
    feature_cols: list[str],
    group_cols: list[str],
    suffix: str = "_z"
) -> pd.DataFrame:
    """
    Add z-score columns for each feature in feature_cols, computed within group_cols.

    Example
    -------
    group_cols = ["League_Name", "age_group"]
    For each KPI 'Passes', you will get:
      - 'Passes_z' = z-score within each (League_Name, age_group)
    """
    df = df.copy()
    for col in feature_cols:
        zcol = f"{col}{suffix}"
        df[zcol] = df.groupby(group_cols)[col].transform(z_score)
    return df

In [7]:
# =============================================================================
# SECTION 3 — FILE I/O HELPERS (DATASET_PER_POSITION)
# =============================================================================

In [8]:
def list_position_csv_files(dataset_dir: Path) -> list[Path]:
    """
    Return all position_*.csv files in dataset_dir, excluding *_removed.csv.

    Example
    -------
    dataset_dir contains:
      - position_8.csv
      - position_8_removed.csv

    Only position_8.csv will be returned.
    """
    files = sorted(dataset_dir.glob("position_*.csv"))
    files = [f for f in files if not f.name.endswith("_removed.csv")]
    return files


def position_name_from_filename(file_path: Path) -> str:
    """
    Extract the position name from the filename:
      - position_Full_back.csv   -> "Full back"
      - position_Centre_back.csv -> "Centre back"
      - position_8.csv           -> "8"

    Why this matters
    ----------------
    You want a clean 'position_name' column to:
      - keep all positions in one master table
      - normalize per position (different KPI sets per position)
      - debug and filter easily in analysis tools
    """
    name = file_path.stem
    name = name.replace("position_", "")
    name = name.replace("_", " ")
    return name.strip()


def read_position_csv(file_path: Path) -> pd.DataFrame:
    """
    Read a single position CSV and add a 'position_name' column.
    """
    df = pd.read_csv(file_path)
    df["position_name"] = position_name_from_filename(file_path)
    return df


def load_all_positions(dataset_dir: Path) -> pd.DataFrame:
    """
    Load all position datasets (excluding *_removed.csv) and concatenate them row-wise.
    """
    files = list_position_csv_files(dataset_dir)
    dfs = [read_position_csv(f) for f in files]
    if not dfs:
        return pd.DataFrame()
    return pd.concat(dfs, ignore_index=True)

In [9]:
# =============================================================================
# SECTION 4 — BUSINESS RULE: MINUTES THRESHOLD FILTERING
# =============================================================================

In [10]:
def add_minutes_threshold_and_filter(
    df: pd.DataFrame,
    minutes_col: str,
    peer_group_cols: list[str],
    stage_minutes: dict,
    minutes_percentile_q: float,
    stage_col: str = "player_stage",
    keep_threshold_col: bool = False,
) -> pd.DataFrame:
    """
    Filter rows based on minutes played per peer group.

    For each row:
      - required_minutes = STAGE_MINUTES[player_stage]
      - quantile_minutes = q-quantile of minutes within peer_group_cols
      - threshold = max(required_minutes, quantile_minutes)

    Keep rows where Minutes >= threshold.

    Why this design?
    ----------------
    - stage-based minimum ensures younger/older stages have different minimum exposure
    - quantile-based minimum protects against weird distributions per league/age group

    Example
    -------
    Suppose player_stage = "Developing Talent" => required_minutes = 300
    In a specific (League, age_group) group, the 10th percentile minutes = 450

    threshold = max(300, 450) = 450
    So we only keep players with Minutes >= 450 in that peer group.
    """
    df = df.copy()

    df[minutes_col] = pd.to_numeric(df[minutes_col], errors="coerce")
    df["_required_minutes"] = df[stage_col].map(stage_minutes)

    df["_quantile_minutes"] = df.groupby(peer_group_cols)[minutes_col].transform(
        lambda s: s.quantile(minutes_percentile_q)
    )

    df["_minutes_threshold"] = df[["_required_minutes", "_quantile_minutes"]].max(axis=1)

    out = df[df[minutes_col] >= df["_minutes_threshold"]].copy()

    if not keep_threshold_col:
        out.drop(columns=["_required_minutes", "_quantile_minutes", "_minutes_threshold"], inplace=True, errors="ignore")

    return out

In [11]:
# =============================================================================
# SECTION 5 — POSITION-SPECIFIC NORMALIZATION (KPI SELECTION + IMPUTE + Z)
# =============================================================================

In [12]:
def get_position_features(position_name: str, position_kpis: dict, df: pd.DataFrame) -> list[str]:
    """
    Return the KPI feature columns for a given position, filtered to those that exist in df.
    Preserves the original order and removes duplicates.

    Why this matters
    ----------------
    Your POSITION_KPIS lists contain placeholders / comments / KPIs that may not exist
    in every CSV. This function makes the pipeline robust.

    Example
    -------
    POSITION_KPIS["8"] might include "Expected Threat (xT)"
    If df doesn't have that column, it will be ignored automatically.
    """
    feats = position_kpis.get(position_name, [])
    feats = [c for c in feats if c in df.columns]

    seen = set()
    feats_unique = []
    for c in feats:
        if c not in seen:
            seen.add(c)
            feats_unique.append(c)

    return feats_unique


def normalize_position_block(
    df_pos: pd.DataFrame,
    position_name: str,
    position_kpis: dict,
    peer_group_cols: list[str],
) -> tuple[pd.DataFrame, dict[str, Any]]:
    """
    Normalize one position-specific dataframe:
      1) determine KPI columns for this position
      2) convert to numeric
      3) capture missingness BEFORE imputation (for sanity reporting)
      4) impute missing values (peer-group medians + fallback median)
      5) capture missingness AFTER imputation
      6) compute z-scores within peer groups

    Returns
    -------
    (df_normalized, stats_dict)
    stats_dict is used by the sanity check report.
    """
    df_pos = df_pos.copy()

    feature_cols = get_position_features(position_name, position_kpis, df_pos)
    stats: dict[str, Any] = {
        "position_name": position_name,
        "n_rows": len(df_pos),
        "n_features": len(feature_cols),
        "features": feature_cols,
        "missing_before": {},
        "missing_after": {},
        "z_cols_added": 0,
    }

    if not feature_cols:
        return df_pos, stats

    df_pos = force_numeric(df_pos, feature_cols)

    # Missingness before imputation
    stats["missing_before"] = df_pos[feature_cols].isna().sum().to_dict()

    df_pos = impute_peer_group_median(df_pos, feature_cols, peer_group_cols)

    # Missingness after imputation
    stats["missing_after"] = df_pos[feature_cols].isna().sum().to_dict()

    df_pos = add_peer_group_zscores(df_pos, feature_cols, peer_group_cols, suffix="_z")
    stats["z_cols_added"] = len(feature_cols)

    return df_pos, stats


def normalize_all_positions(
    df: pd.DataFrame,
    position_kpis: dict,
    peer_group_cols: list[str],
    position_col: str = "position_name",
) -> tuple[pd.DataFrame, list[dict[str, Any]]]:
    """
    Split df by position, normalize each block using its own KPI list,
    then concatenate everything back together.

    Returns
    -------
    (df_normalized, per_position_stats)
    """
    out_parts = []
    stats_all: list[dict[str, Any]] = []

    for pos_name, df_pos in df.groupby(position_col, dropna=False):
        df_norm, stats = normalize_position_block(df_pos, str(pos_name), position_kpis, peer_group_cols)
        out_parts.append(df_norm)
        stats_all.append(stats)

    df_out = pd.concat(out_parts, ignore_index=True) if out_parts else df.copy()
    return df_out, stats_all

In [13]:
# =============================================================================
# SECTION 6 — SANITY CHECK REPORTING (DEBUG-FRIENDLY)
# =============================================================================

In [14]:
def _rows_by_position(df: pd.DataFrame, position_col: str) -> dict[str, int]:
    """
    Helper: return row counts by position as a simple dict.
    """
    if df.empty or position_col not in df.columns:
        return {}
    return df[position_col].value_counts(dropna=False).to_dict()


def build_sanity_report(
    df_before_filter: pd.DataFrame,
    df_after_filter: pd.DataFrame,
    normalization_stats: list[dict[str, Any]],
    peer_group_cols: list[str],
    position_col: str = "position_name",
) -> dict[str, Any]:
    """
    Build a structured sanity check report to make debugging and delivery easier.

    Report includes
    --------------
    - Row counts before/after minutes filtering (overall + per position)
    - Peer group coverage after filtering
    - Missingness before/after imputation (per position, per KPI)
    - How many z-score columns were added per position

    Returns
    -------
    A dictionary that you can:
      - print nicely
      - log
      - export to JSON
      - show in a notebook
    """
    report: dict[str, Any] = {}

    report["rows_total_before_minutes_filter"] = len(df_before_filter)
    report["rows_total_after_minutes_filter"] = len(df_after_filter)

    report["rows_by_position_before_filter"] = _rows_by_position(df_before_filter, position_col)
    report["rows_by_position_after_filter"] = _rows_by_position(df_after_filter, position_col)

    # Peer group coverage after filtering
    if not df_after_filter.empty:
        report["n_peer_groups_after_filter"] = df_after_filter[peer_group_cols].drop_duplicates().shape[0]
        report["peer_groups_cols"] = peer_group_cols
    else:
        report["n_peer_groups_after_filter"] = 0
        report["peer_groups_cols"] = peer_group_cols

    # Normalization details per position
    per_pos = {}
    for s in normalization_stats:
        pos = s["position_name"]
        per_pos[pos] = {
            "n_rows": s.get("n_rows", 0),
            "n_features_normalized": s.get("n_features", 0),
            "z_cols_added": s.get("z_cols_added", 0),
            "missing_before_sum": int(sum(s.get("missing_before", {}).values())),
            "missing_after_sum": int(sum(s.get("missing_after", {}).values())),
            # Keep full per-feature counts for deep debugging
            "missing_before_by_feature": s.get("missing_before", {}),
            "missing_after_by_feature": s.get("missing_after", {}),
        }

    report["normalization_by_position"] = per_pos

    return report


def print_sanity_report(report: dict[str, Any]) -> None:
    """
    Pretty-print the sanity report in a notebook-friendly way.
    """
    print("=== SANITY CHECK REPORT ===")
    print(f"Rows before minutes filter: {report.get('rows_total_before_minutes_filter')}")
    print(f"Rows after minutes filter:  {report.get('rows_total_after_minutes_filter')}")
    print()

    print("Rows by position (before -> after):")
    before = report.get("rows_by_position_before_filter", {})
    after = report.get("rows_by_position_after_filter", {})
    all_pos = sorted(set(before.keys()) | set(after.keys()), key=str)

    for p in all_pos:
        print(f"  - {p}: {before.get(p, 0)} -> {after.get(p, 0)}")

    print()
    print(f"Peer groups after filter: {report.get('n_peer_groups_after_filter')}  (cols: {report.get('peer_groups_cols')})")
    print()

    print("Normalization (missing values before/after imputation):")
    norm = report.get("normalization_by_position", {})
    for pos, info in norm.items():
        print(f"  - {pos}: features={info['n_features_normalized']}, z_added={info['z_cols_added']}, "
              f"missing_sum {info['missing_before_sum']} -> {info['missing_after_sum']}")

In [15]:
# =============================================================================
# SECTION 7 — PIPELINE ENTRYPOINT (ONE FUNCTION TO RUN EVERYTHING)
# =============================================================================

In [16]:
@dataclass
class PipelineResult:
    """
    Container for pipeline output + debug report + output paths (if saved).
    """
    df_master: pd.DataFrame
    sanity_report: dict[str, Any]
    output_csv_path: Path | None = None
    output_xlsx_path: Path | None = None
    output_report_path: Path | None = None


def build_normalized_master_table(
    project_root: Path,
    position_kpis: dict,
    age_groups: list[str],
    stage_by_agegroup: dict,
    stage_minutes: dict,
    minutes_percentile_q: float,
    minutes_col: str,
    peer_group_cols: list[str],
    input_subdir: str = "dataset_per_position",
    age_col: str = "Age",
    position_col: str = "position_name",
    # --- Output options ---
    save_output: bool = False,
    output_subdir: str = "normalized_dataset",
    output_filename: str = "normalized_players.csv",
    save_excel: bool = False,
    excel_filename: str = "normalized_players.xlsx",
    save_report_json: bool = False,
    report_filename: str = "sanity_report.json",
    verbose: bool = True,
) -> PipelineResult:
    """
    End-to-end pipeline:
      1) Load all position CSVs (excluding *_removed.csv) from input_subdir
      2) Add age_group and player_stage
      3) Filter rows based on minutes thresholds within peer groups
      4) Normalize features per position:
         - numeric conversion
         - peer-group median imputation
         - z-scores per peer group
      5) Produce a sanity check report for debugging + delivery
      6) Optionally save outputs (CSV and/or Excel) + report to output_subdir

    Notes on Excel export
    ---------------------
    Writing .xlsx requires the 'openpyxl' package.
    If save_excel=True but openpyxl is not installed, Excel export is skipped
    and output_xlsx_path remains None.
    """
    input_dir = project_root / input_subdir

    # 1) Load
    df_all = load_all_positions(input_dir)

    if df_all.empty:
        raise ValueError(
            f"No position CSV files found in: {input_dir}\n"
            "Expected files like: position_8.csv, position_Full_back.csv, ..."
        )

    # Optional but recommended: strip column names to avoid hidden whitespace bugs
    df_all.columns = df_all.columns.str.strip()

    # 2) Add age_group & stage
    df_all = add_age_group_and_stage(
        df_all,
        age_col=age_col,
        age_groups=age_groups,
        stage_by_agegroup=stage_by_agegroup,
        out_age_group_col="age_group",
        out_stage_col="player_stage",
    )

    # 3) Minutes filter (keep a copy for sanity reporting)
    df_before_filter = df_all.copy()
    df_filtered = add_minutes_threshold_and_filter(
        df_all,
        minutes_col=minutes_col,
        peer_group_cols=peer_group_cols,
        stage_minutes=stage_minutes,
        minutes_percentile_q=minutes_percentile_q,
        stage_col="player_stage",
        keep_threshold_col=False,
    )

    # 4) Normalize per position
    df_norm, stats = normalize_all_positions(
        df_filtered,
        position_kpis=position_kpis,
        peer_group_cols=peer_group_cols,
        position_col=position_col,
    )

    # 5) Sanity report
    report = build_sanity_report(
        df_before_filter=df_before_filter,
        df_after_filter=df_filtered,
        normalization_stats=stats,
        peer_group_cols=peer_group_cols,
        position_col=position_col,
    )

    # 6) Optional: save outputs
    output_csv_path = None
    output_xlsx_path = None
    output_report_path = None

    if save_output or save_excel or save_report_json:
        output_dir = project_root / output_subdir
        output_dir.mkdir(parents=True, exist_ok=True)

        if save_output:
            output_csv_path = output_dir / output_filename
            df_norm.to_csv(output_csv_path, index=False)

        if save_excel:
            # Excel requires openpyxl. If not installed, skip with a friendly message.
            try:
                import openpyxl  # noqa: F401
                output_xlsx_path = output_dir / excel_filename
                df_norm.to_excel(output_xlsx_path, index=False)
            except ModuleNotFoundError:
                output_xlsx_path = None
                if verbose:
                    print(
                        "Excel export skipped: 'openpyxl' is not installed.\n"
                        "Install with: pip install openpyxl  (or conda install openpyxl)"
                    )

        if save_report_json:
            import json
            output_report_path = output_dir / report_filename
            with open(output_report_path, "w", encoding="utf-8") as f:
                json.dump(report, f, indent=2)

    return PipelineResult(
        df_master=df_norm,
        sanity_report=report,
        output_csv_path=output_csv_path,
        output_xlsx_path=output_xlsx_path,
        output_report_path=output_report_path,
    )

In [17]:
# =============================================================================
# SECTION 8 — NORMALIZED DATASET RESULT
# =============================================================================

In [22]:
result = build_normalized_master_table(
    project_root=project_root,
    position_kpis=POSITION_KPIS,
    age_groups=AGE_GROUPS,
    stage_by_agegroup=STAGE_BY_AGEGROUP,
    stage_minutes=STAGE_MINUTES,
    minutes_percentile_q=MINUTES_PERCENTILE_Q,
    minutes_col=MINUTES_COLUMN,
    peer_group_cols=PEER_GROUP_COLS,
    input_subdir="dataset_per_position",     # input
    age_col="Age",
    save_output=True,                        # save CSV
    output_subdir="normalized_dataset",
    output_filename="normalized_players.csv",
    save_excel=True,                         # save XLSX (if openpyxl is available)
    excel_filename="normalized_players.xlsx",
    save_report_json=True,
    report_filename="sanity_report.json",
)

df_master = result.df_master
report = result.sanity_report

print_sanity_report(report)
df_master.head()

print("Saved CSV to:", result.output_csv_path)
print("Saved XLSX to:", result.output_xlsx_path)  # None if openpyxl missing
print("Saved report to:", result.output_report_path)

=== SANITY CHECK REPORT ===
Rows before minutes filter: 16253
Rows after minutes filter:  13848

Rows by position (before -> after):
  - 10: 3013 -> 2633
  - 6: 3000 -> 2518
  - 7: 1250 -> 1041
  - 8: 2769 -> 2344
  - 9: 1834 -> 1578
  - Centre back: 2429 -> 2081
  - Full back: 1958 -> 1653

Peer groups after filter: 36  (cols: ['League_Name', 'age_group'])

Normalization (missing values before/after imputation):
  - 10: features=31, z_added=31, missing_sum 2802 -> 0
  - 6: features=30, z_added=30, missing_sum 2893 -> 0
  - 7: features=33, z_added=33, missing_sum 913 -> 0
  - 8: features=34, z_added=34, missing_sum 1104 -> 0
  - 9: features=33, z_added=33, missing_sum 1609 -> 0
  - Centre back: features=33, z_added=33, missing_sum 693 -> 0
  - Full back: features=36, z_added=36, missing_sum 903 -> 0
Saved CSV to: C:\Users\tijn0\OneDrive\Documenten\GitHub\business_analytics_workshop\normalized_dataset\normalized_players.csv
Saved XLSX to: C:\Users\tijn0\OneDrive\Documenten\GitHub\busine

In [19]:
# TESTS -> will be removed later

In [23]:
import pandas as pd
from pathlib import Path

# --- 1) Load the normalized dataset (avoid dtype warning) ---
csv_path = project_root / "normalized_dataset" / "normalized_players.csv"
df = pd.read_csv(csv_path, low_memory=False)

# --- 2) Filter to Centre back ---
df_cb = df[df["position_name"] == "Centre back"].copy()

# --- 3) Metadata columns to keep ---
metadata_cols = [
    "Name", "Team", "Position", "Nationality", "Birth", "Season",
    "(€) Market Value", "Contract End", "Age", "Height (cm)",
    "Side", "Strong foot", "GBE", "Games", "Minutes", "Yellow Cards",
    "Excelsior Score", "Transfermarkt_Value", "Transfermarkt_Player_ID",
    "Country", "Division", "League_Name", "League_Quality",
    "Player_Quality", "Pricing", "position_name", "age_group", "player_stage"
]
metadata_cols = [c for c in metadata_cols if c in df_cb.columns]

# --- 4) Centre back features (raw + z) ---
features = [
    "% Passing Contribution",
    "xG Build Up",
    "Ball Prog. by Passing",
    "% Aerial Wins Own Box",
    "Tackles/Was Dribbled",
    "Interceptions",
    "Clearances",
]

cols_to_show = []
missing_features = []

for f in features:
    if f not in df_cb.columns:
        missing_features.append(f)
        continue

    cols_to_show.append(f)

    zcol = f"{f}_z"
    if zcol in df_cb.columns:
        cols_to_show.append(zcol)
    else:
        missing_features.append(zcol)

# --- 5) Final view ---
df_cb_view = df_cb[metadata_cols + cols_to_show].copy()

# --- 6) Sort by desired z-score ---
sort_col = "% Aerial Wins Own Box_z"

if sort_col in df_cb_view.columns:
    df_cb_view = df_cb_view.sort_values(sort_col, ascending=False)
else:
    print(f"Sort column not found: {sort_col}")

# --- 7) Output ---
print("Rows (Centre back):", len(df_cb_view))

if missing_features:
    print("\nMissing columns:")
    for m in missing_features:
        print(" -", m)

# --- 8) Export paths ---
export_dir = project_root / "normalized_dataset" / "tests"
export_dir.mkdir(parents=True, exist_ok=True)

csv_export_path = export_dir / "centre_back_feature_test.csv"
xlsx_export_path = export_dir / "centre_back_feature_test.xlsx"

# --- 9) Save ---
df_cb_view.to_csv(csv_export_path, index=False)
df_cb_view.to_excel(xlsx_export_path, index=False)

print("Exported test datasets:")
print("CSV :", csv_export_path)
print("XLSX:", xlsx_export_path)


Rows (Centre back): 2081
Exported test datasets:
CSV : C:\Users\tijn0\OneDrive\Documenten\GitHub\business_analytics_workshop\normalized_dataset\tests\centre_back_feature_test.csv
XLSX: C:\Users\tijn0\OneDrive\Documenten\GitHub\business_analytics_workshop\normalized_dataset\tests\centre_back_feature_test.xlsx
